In [2]:
import os
from huggingface_hub import snapshot_download

model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" 
save_dir = os.path.join(os.getcwd(), "models", "Qwen2.5-3B-Instruct-unsloth-4bit")

os.makedirs(save_dir, exist_ok=True)
print(f"Model will be downloaded: {save_dir}")

if os.path.exists(save_dir) and os.listdir(save_dir):
    print("Model already exists, skipping installation.")
else:
    snapshot_download(
        repo_id=model_name,
        local_dir=save_dir,
        local_dir_use_symlinks=False
    )
    print("\nDownloaded successfully!")

print(f"Files: {os.listdir(save_dir)}")

Model will be downloaded: /home/yesoytur/APilus/models/Qwen2.5-3B-Instruct-unsloth-4bit


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]


Downloaded successfully!
Files: ['.cache', 'generation_config.json', 'config.json', '.gitattributes', 'vocab.json', 'added_tokens.json', 'special_tokens_map.json', 'model.safetensors', 'tokenizer_config.json', 'merges.txt', 'tokenizer.json', 'README.md']


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
import os
import time

model_path = os.path.join(os.getcwd(), "models", "Qwen2.5-3B-Instruct-unsloth-4bit")

tokenizer = AutoTokenizer.from_pretrained(model_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Model loaded!")

/home/yesoytur/APilus/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model loaded!


In [4]:
def ask(
    prompt,
    enable_thinking=False,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.1,
    do_sample=True,
    seed=None,
    verbose=True,
):
    if seed is not None:
        import torch
        torch.manual_seed(seed)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_token_count = model_inputs.input_ids.shape[1]

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        top_k=top_k if do_sample else None,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )

    start_time = time.perf_counter()
    generated_ids = model.generate(**model_inputs, generation_config=gen_config)
    elapsed = time.perf_counter() - start_time

    output_ids = generated_ids[0][input_token_count:]
    output_token_count = len(output_ids)
    tokens_per_second = output_token_count / elapsed

    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    if verbose:
        print(f"\n{'─'*48}")
        print(f"  🌡  Temperature         : {temperature}")
        print(f"  🎯 Top-p               : {top_p}")
        print(f"  🔝 Top-k               : {top_k}")
        print(f"  🔁 Repetition penalty  : {repetition_penalty}")
        print(f"  🎲 Sampling            : {do_sample}  |  Seed: {seed}")
        print(f"  {'─'*42}")
        print(f"  📥 Input tokens        : {input_token_count:>6} tokens")
        print(f"  📤 Output tokens       : {output_token_count:>6} tokens")
        print(f"  ⏱  Generation time     : {elapsed:>6.2f} s")
        print(f"  ⚡ Tokens per second   : {tokens_per_second:>6.1f} tok/s")
        print(f"{'─'*48}\n")

    return response

print("✅ ask() ready!")

✅ ask() ready!


In [ ]:
print(ask("Who is Acıbadem Universitys Computer Engineering Department Head?", seed=42))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.7
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: 42
  ──────────────────────────────────────────
  📥 Input tokens        :     42 tokens
  📤 Output tokens       :     79 tokens
  ⏱  Generation time     :   2.49 s
  ⚡ Tokens per second   :   31.7 tok/s
────────────────────────────────────────────────

I'm sorry for the confusion earlier, but as an AI, I don't have real-time access to update information about specific individuals or institutions. However, you can easily find this information on the official website of Acıbadem University or through their direct communication channels.

To get the most accurate and up-to-date information, I recommend visiting the university's official website or contacting them directly.


In [9]:
print(ask("Acıbadem Üniversitesi Bilgisayar Mühendisliği Bölüm başkanı kimdir?", temperature=0.6, seed=42))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.6
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: 42
  ──────────────────────────────────────────
  📥 Input tokens        :     50 tokens
  📤 Output tokens       :    103 tokens
  ⏱  Generation time     :   3.29 s
  ⚡ Tokens per second   :   31.3 tok/s
────────────────────────────────────────────────

Üzümsel bir bilgi vermektedir ki, şu anda Acıbadem Üniversitesi Bilgisayar Mühendisliği Bölüm başkanının adını veya ismini bilmeyim. Bu konuda en güncel ve doğrudan kaynağına başvurulması gereken bilgiler olmalıdır. Örneğin, üniversite resmi sayfalarında veya mevcut görevler listelerinde arama yaparak bu bilgiyi bulabilirsiniz.


In [10]:
print(ask("En iyi Aİ hangisidir?", temperature=0.6, seed=42))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.6
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: 42
  ──────────────────────────────────────────
  📥 Input tokens        :     39 tokens
  📤 Output tokens       :    289 tokens
  ⏱  Generation time     :   8.78 s
  ⚡ Tokens per second   :   32.9 tok/s
────────────────────────────────────────────────

Bu konuda tartışmak ve bir seçim yapmak zordur çünkü her sistemin farklı özelliklerine sahip olup, uygun durumlarda birinin diğerinden daha iyi sonuç vermesi beklenir. İleri teknolojiler arasında AI sistemi seçmek karmaşıktır ve genel olarak belirtmek kolay değildir.

Ama bazı önde gitmekte olan ve popüler AI sistemleri şunlar:

1. Anthropic'nin Claude
2. Google'in LaMDA (Language Model for Dialogue Applications)
3. Anthropic'nin Claude'dan daha genç bir versiyon olan Claude 2
4. Anthropic'nin başka bir AI sohbet botu - Claude 2'd

In [11]:
print(ask("Opus4.6 modeli kaç parametreden oluşuyor", temperature=0.6, seed=42))


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.6
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: 42
  ──────────────────────────────────────────
  📥 Input tokens        :     44 tokens
  📤 Output tokens       :    177 tokens
  ⏱  Generation time     :   5.55 s
  ⚡ Tokens per second   :   31.9 tok/s
────────────────────────────────────────────────

Üzgünüm ama benim için Opus 4.6 modelinin tamamen güncel ve detaylı parametre sayısını bilmeme imkan yok. Modelin genel boyutu hakkında belgeler veya bilgilere ulaşmak için, Opus projektine ait resmi kaynakları veya ilgili araştırma makalelerine başvurmanızı öneririm. Bunun yerine, genellikleopus modelleri genellikle çok büyük sayıda parametre içerir ve 4.6 sürümünde da olsa belki 100.000'lük veya daha fazla parametre olabilir. Herhangi bir modelin tam parametre sayısı hakkında kesin bilgi alabilmek için projeye ait resmi kaynakla

In [15]:
print(ask("merika İrana saldırdı mı?", temperature=0.8, seed=42))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.8
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: 42
  ──────────────────────────────────────────
  📥 Input tokens        :     40 tokens
  📤 Output tokens       :    253 tokens
  ⏱  Generation time     :   7.82 s
  ⚡ Tokens per second   :   32.3 tok/s
────────────────────────────────────────────────

İşte şu bilgilere dayanarak daha fazla bilgi edinebilirsiniz:

1. Mücadele: Amerika Birleşik Devletleri ile Irak'ın arasında bir muharebe saptadığımızda, bu muharebeler 2003 yılında başlamıştır.

2. Gelecek Saldır: Amerika İrak'yı işte 2003 yılının mart ayında ilk saldiri faustetleriyle başlatmıştır.

3. Bağımsızlık Tarihinde: Bu tür saldırılar, Irak'nın bağımsızlığına olanak tanıyan ve onun yenilenmesini sağlayacak bir meydan okuması anlamına gelir.

4. Çarpımlar: Amerika'nın çarpımları İran'a karşı da kullanılması mümkündür, anc